In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================
# 1. GERANDO O DATASET (Simulação de Negócio)
# ==========================================
np.random.seed(42)
n_usuarios = 1000

data = {
    'ID_Usuario': range(1, n_usuarios + 1),
    'Idade': np.random.randint(18, 65, n_usuarios),
    'Mensalidade': np.random.choice([29.90, 45.90, 55.90], n_usuarios),
    'Meses_Contrato': np.random.randint(1, 24, n_usuarios),
    'Acessos_Mes': np.random.randint(0, 30, n_usuarios),
    'Suporte_Aberto': np.random.randint(0, 5, n_usuarios)
}

df = pd.DataFrame(data)

# Criando a lógica de Churn (Alvo do modelo)
# Regra: Se acessa pouco (< 8) ou tem muito suporte (> 3), a chance de cancelar é alta
df['Churn'] = ((df['Acessos_Mes'] < 8) | (df['Suporte_Aberto'] > 3)).astype(int)

print("✅ Dataset criado com sucesso!")
print(df.head())

# ==========================================
# 2. ANÁLISE EXPLORATÓRIA (EDA)
# ==========================================
# Matriz de Correlação
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de Correlação de Variáveis')
plt.show()



# Boxplot: Acessos vs Churn
plt.figure(figsize=(8, 5))
sns.boxplot(x='Churn', y='Acessos_Mes', data=df, palette='Set2')
plt.title('Distribuição de Acessos: Clientes que ficaram vs Clientes que saíram')
plt.show()

# ==========================================
# 3. PREPARAÇÃO PARA MACHINE LEARNING
# ==========================================
# X = O que o robô vê (Características) | y = O que o robô prevê (Churn)
X = df[['Idade', 'Mensalidade', 'Meses_Contrato', 'Acessos_Mes', 'Suporte_Aberto']]
y = df['Churn']

# Dividindo 80% para treino e 20% para teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 4. TREINAMENTO DO MODELO (Árvore de Decisão)
# ==========================================
modelo = DecisionTreeClassifier(max_depth=3) # Limitamos a profundidade para ser fácil de explicar
modelo.fit(X_train, y_train)

# Fazendo previsões com os dados de teste
previsoes = modelo.predict(X_test)

# ==========================================
# 5. AVALIAÇÃO DE RESULTADOS
# ==========================================
acuracia = accuracy_score(y_test, previsoes)
print(f"\n🎯 Acurácia do Modelo: {acuracia * 100:.2f}%")

print("\n📋 Relatório de Classificação:")
print(classification_report(y_test, previsoes))

# Visualizando a "Lógica" do Robô
plt.figure(figsize=(20, 10))
plot_tree(modelo, feature_names=X.columns, class_names=['Fica', 'Cancela'], filled=True)
plt.title('Árvore de Decisão para Cancelamento de Assinatura')
plt.show()


